# 06 · Paper figures and tables (all datasets)

Builds the figures and tables from the files produced by the other notebooks (`results/…`, `esp32/…`).
No raw data and no model are needed; it runs in seconds.

Outputs: `paper/figures/*.png|pdf` (300 dpi + vector) and `paper/tables/*.csv|tex` (LaTeX, booktabs).

| Figure | Content |
|---|---|
| fig01 | Framework overview |
| fig02 | Class distribution (log scale) of the held-out test split per dataset |
| fig03 | Headline test metrics per dataset (accuracy, MCC, macro-F1, minority recall) |
| fig04 | Selected tree vs baselines: MCC and size |
| fig05 | Per-class recall and F1 |
| fig06 | Row-normalised confusion matrices |
| fig07 | Feature-count (K) and pruning ablation |
| fig08 | Multi-seed robustness (mean ± std) |
| fig09 | Search convergence (if notebook 01 was run) |
| fig10 | Tuning methods at equal budget (if notebook 04 was run) |
| fig11 | Class-weight ablation (if notebook 02 was run) |
| hw_* | ESP32 figures from notebook 05 |

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "paper_utils.py").exists():
    if ROOT.parent == ROOT: raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display
from paper_utils import *
set_paper_style()
RES, FIG, TAB = ROOT / "results", ROOT / "paper" / "figures", ROOT / "paper" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)

def have(ds, *parts): return (RES.joinpath(*parts[:1], ds, *parts[1:])).exists()
present = [d for d in DATASETS if have(d, "article_results", "article_results_summary.json")]
print("datasets with results:", [DS_LABEL[d] for d in present])
S = {d: load_json(RES / "article_results" / d / "article_results_summary.json") for d in present}
PC = {d: read_csv_if(RES / "publishability_checks" / d / "per_class_metrics_saved_model.csv") for d in present}
BL = {d: read_csv_if(RES / "publishability_checks" / d / "baseline_comparison_exact_split.csv") for d in present}

## Tables

In [ ]:
# T1 dataset summary
t1 = pd.DataFrame([{"Dataset": DS_LABEL[d], "Rows": S[d]["n_total"], "Classes": len(S[d]["class_names"]),
                    "Train": S[d]["n_train"], "Val": S[d]["n_validation"], "Test": S[d]["n_test"],
                    "Selected K": S[d]["validation_selected_k"],
                    "Largest class %": 100 * PC[d]["support"].max() / PC[d]["support"].sum(),
                    "Smallest class (test n)": int(PC[d]["support"].min())} for d in present])
display(t1); save_table(t1, TAB / "tab01_datasets", "Dataset profiles and split sizes.", "tab:datasets", "{:.2f}")

# T2 main held-out results (+ multi-seed mean±std where available)
rows = []
for d in present:
    m = S[d]["held_out_test_metrics"]
    ms = read_csv_if(RES / "article_results" / d / "multi_seed_repeats_summary.csv")
    def pm(k):
        if ms is None: return ""
        r = ms[ms.iloc[:, 0] == k]
        return f"{float(r['mean'].iloc[0]):.4f} ± {float(r['std'].iloc[0]):.4f}" if len(r) else ""
    rows.append({"Dataset": DS_LABEL[d], "Accuracy": m["accuracy"], "Macro-F1": m["macro_f1"], "Weighted-F1": m["weighted_f1"],
                 "MCC": m["mcc"], "Kappa": m["kappa"], "Small-class F1": m["small_class_f1"], "Minority recall": m["minority_recall"],
                 "Rare-class MCC": m["rare_class_mcc"], "Nodes": int(m["node_count"]), "Depth": int(m["depth"]),
                 "Flash est. (KB)": flash_kb(m["node_count"]), "MCC mean±std (seeds)": pm("mcc"), "Macro-F1 mean±std (seeds)": pm("macro_f1")})
t2 = pd.DataFrame(rows); display(t2)
save_table(t2, TAB / "tab02_main_results", "Held-out test results of the optimized decision tree.", "tab:main", "{:.4f}")

In [ ]:
# T3 baselines (same split) -- one long table + a compact 'selected models' version
frames = []
for d in present:
    b = BL[d].copy(); b.insert(0, "Dataset", DS_LABEL[d]); frames.append(b)
t3 = pd.concat(frames)[["Dataset", "model", "accuracy", "macro_f1", "mcc", "node_count", "size_proxy_kb"]]
t3.columns = ["Dataset", "Model", "Accuracy", "Macro-F1", "MCC", "Nodes", "Flash est. (KB)"]
save_table(t3, TAB / "tab03_baselines_full", "Same-split baseline comparison (all models).", "tab:baselines_full", "{:.4f}")
keep = ["Optimized_DT_HSDO_saved", "DT_default_all_features", "DT_default_best_k", "DT_default_all_features_FULLTRAIN", "DT_default_best_k_FULLTRAIN",
        "RandomForest_100_best_k", "ExtraTrees_100_best_k", "XGBoost_200_best_k", "KNN_5_best_k", "LogReg_balanced_best_k"]
t3c = t3[t3.Model.isin(keep)]
display(t3c); save_table(t3c, TAB / "tab03_baselines_selected", "Optimized DT vs selected baselines on the identical held-out split.", "tab:baselines", "{:.4f}")

# T4 ablation, T5 per-class
fr = []
for d in present:
    a = read_csv_if(RES / "publishability_checks" / d / "ablation_study.csv")
    if a is not None: a = a.copy(); a.insert(0, "Dataset", DS_LABEL[d]); fr.append(a[["Dataset", "model", "accuracy", "macro_f1", "mcc", "node_count", "depth", "size_proxy_kb"]])
t4 = pd.concat(fr); t4.columns = ["Dataset", "Variant", "Accuracy", "Macro-F1", "MCC", "Nodes", "Depth", "Flash est. (KB)"]
save_table(t4, TAB / "tab04_ablation", "Ablation: number of selected features and pruning.", "tab:ablation", "{:.4f}")
t5 = pd.concat([PC[d].assign(Dataset=DS_LABEL[d])[["Dataset", "class_name", "support", "precision", "recall", "f1"]] for d in present])
t5.columns = ["Dataset", "Class", "Test support", "Precision", "Recall", "F1"]
save_table(t5, TAB / "tab05_per_class", "Per-class test metrics of the optimized decision tree.", "tab:perclass", "{:.4f}")
display(t5[t5["Recall"] < 0.05])

## Figures

In [ ]:
# fig01 -- framework overview
fig, ax = plt.subplots(figsize=(7.2, 2.7)); ax.axis("off"); ax.set_xlim(0, 100); ax.set_ylim(0, 40); ax.grid(False)
def _box(x, y, w, h, t, fc, ec):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.25,rounding_size=1", fc=fc, ec=ec, lw=0.9))
    ax.text(x + w / 2, y + h / 2, t, ha="center", va="center", fontsize=6.8, linespacing=1.25)
def _arrow(x1, y1, x2, y2, c):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=8, color=c, lw=0.9))
top = ["Dataset profile\ntarget, dropped\nidentifiers", "Leakage-aware\npreprocessing:\nmedian imputing,\nrobust scaling",
       "SelectKBest\nK chosen on\nvalidation", "Tree tuning\nrestricted space,\nvalidation MCC,\nsize limit", "Refit on\ntrain + val;\ntest once"]
bot = ["Exact export:\nraw-space thresholds\nby float32 bisection,\n10 B per node", "Verification 1\nevery node: R left,\nnext float32 right",
       "Verification 2\ncompiled C on the\nfull test split and\nadversarial cuts", "Verification 3\nESP32 board:\nagreement, latency,\nflash"]
w, h, y = 16.4, 12, 24; xs = [1 + i * 19.8 for i in range(5)]
for x, t in zip(xs, top): _box(x, y, w, h, t, "#f1f0ee", "#52514e")
for p, q in zip(xs[:-1], xs[1:]): _arrow(p + w + 0.4, y + h / 2, q - 0.4, y + h / 2, "#52514e")
xb, wb, yb = [1, 26, 51, 76], 22, 6
for x, t in zip(xb, bot): _box(x, yb, wb, h, t, "#dce9f8", "#2a78d6")
for p, q in zip(xb[:-1], xb[1:]): _arrow(p + wb + 0.4, yb + h / 2, q - 0.4, yb + h / 2, "#2a78d6")
ax.plot([xs[4] + w / 2] * 2, [y - 0.4, 21], color="#52514e", lw=0.9); ax.plot([xs[4] + w / 2, 12], [21, 21], color="#52514e", lw=0.9)
_arrow(12, 21, 12, yb + h + 0.5, "#52514e")
ax.text(1, 38.6, "Model building (training and validation rows only)", fontsize=7.2, color="#52514e", style="italic")
ax.text(1, 2.2, "Deployment and verification", fontsize=7.2, color="#2a78d6", style="italic")
save_fig(fig, FIG / "fig01_framework_pipeline"); plt.show()

# fig02 -- class distribution
fig, axes = plt.subplots(1, len(present), figsize=(4.2 * len(present), 3.3), sharey=False)
axes = np.atleast_1d(axes)
for ax, d in zip(axes, present):
    p = PC[d].sort_values("support", ascending=False)
    ax.bar(p["class_name"], p["support"], color=DS_COLOR[d]); ax.set_yscale("log"); ax.set_title(DS_LABEL[d])
    plt.setp(ax.get_xticklabels(), rotation=60, ha="right"); ax.set_ylabel("test support (log)")
fig.suptitle("Class imbalance in the held-out test splits"); save_fig(fig, FIG / "fig02_class_distribution"); plt.show()

# fig03 -- headline metrics across datasets
mets = [("accuracy", "Accuracy"), ("mcc", "MCC"), ("macro_f1", "Macro-F1"), ("minority_recall", "Minority recall")]
fig, ax = plt.subplots(figsize=(7.5, 3.5)); w = .8 / len(present)
for j, d in enumerate(present):
    m = S[d]["held_out_test_metrics"]; ax.bar(np.arange(len(mets)) + j * w, [m[k] for k, _ in mets], w, label=DS_LABEL[d], color=DS_COLOR[d])
    for i, (k, _) in enumerate(mets): ax.text(i + j * w, m[k] + .01, f"{m[k]:.2f}", ha="center", fontsize=6.5)
ax.set_xticks(np.arange(len(mets)) + w * (len(present) - 1) / 2, [l for _, l in mets]); ax.set_ylim(0, 1.1); ax.legend(loc="lower left")
ax.set_title("Optimized decision tree: held-out test metrics (aggregate accuracy hides minority-class failure)")
save_fig(fig, FIG / "fig03_cross_dataset_metrics"); plt.show()

In [ ]:
# fig04 -- baselines: MCC bars + MCC vs size
fig, axes = plt.subplots(2, len(present), figsize=(4.6 * len(present), 6.4))
axes = np.array(axes).reshape(2, len(present))
for j, d in enumerate(present):
    b = BL[d].dropna(subset=["mcc"]).sort_values("mcc")
    b = b[~b.model.str.contains("FULLTRAIN|Dummy")]
    ax = axes[0, j]; ax.barh(b["model"], b["mcc"], color=[HIGHLIGHT if "Optimized" in m else NEUTRAL for m in b["model"]])
    ax.set_title(DS_LABEL[d]); ax.set_xlabel("test MCC"); ax.tick_params(axis="y", labelsize=6)
    ax = axes[1, j]; t = b.dropna(subset=["size_proxy_kb"])
    ax.scatter(t["size_proxy_kb"], t["mcc"], c=[HIGHLIGHT if "Optimized" in m else NEUTRAL for m in t["model"]], s=45)
    for _, r in t.iterrows():
        if "Optimized" in r["model"] or "default_all" in r["model"]: ax.annotate(r["model"].replace("_best_k", "").replace("_features", ""), (r["size_proxy_kb"], r["mcc"]), fontsize=6, xytext=(3, 3), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("model flash, exact export (KB, log) — single-tree models only"); ax.set_ylabel("test MCC")
fig.suptitle("Optimized DT vs same-split baselines"); save_fig(fig, FIG / "fig04_baselines"); plt.show()

# fig05 -- per-class recall and F1
fig, axes = plt.subplots(1, len(present), figsize=(4.6 * len(present), 3.6)); axes = np.atleast_1d(axes)
for ax, d in zip(axes, present):
    p = PC[d]; x = np.arange(len(p))
    ax.bar(x - .2, p["recall"], .4, label="recall", color=DS_COLOR[d]); ax.bar(x + .2, p["f1"], .4, label="F1", color=DS_COLOR[d], alpha=.45)
    ax.set_xticks(x, p["class_name"], rotation=60, ha="right"); ax.set_ylim(0, 1.05); ax.set_title(DS_LABEL[d]); ax.legend(fontsize=7)
fig.suptitle("Per-class performance (test)"); save_fig(fig, FIG / "fig05_per_class"); plt.show()

# fig06 -- normalised confusion matrices
fig, axes = plt.subplots(1, len(present), figsize=(5.2 * len(present), 4.8)); axes = np.atleast_1d(axes)
for ax, d in zip(axes, present):
    cm = read_csv_if(RES / "publishability_checks" / d / "confusion_matrix_saved_model.csv", index_col=0)
    nm = cm.div(cm.sum(axis=1).replace(0, 1), axis=0)
    im = ax.imshow(nm.values, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
    ax.set_xticks(range(len(cm.columns)), [c.replace("pred_", "") for c in cm.columns], rotation=70, ha="right", fontsize=6.5)
    ax.set_yticks(range(len(cm.index)), [c.replace("true_", "") for c in cm.index], fontsize=6.5)
    for i in range(nm.shape[0]):
        for k in range(nm.shape[1]):
            if nm.values[i, k] >= .05: ax.text(k, i, f"{nm.values[i, k]:.2f}", ha="center", va="center", fontsize=5.5, color="white" if nm.values[i, k] > .5 else "black")
    ax.set_title(DS_LABEL[d]); ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.suptitle("Row-normalised confusion matrices (recall per true class)"); save_fig(fig, FIG / "fig06_confusion_matrices"); plt.show()

In [ ]:
# fig07 -- ablation trade-off
fig, axes = plt.subplots(1, len(present), figsize=(4.6 * len(present), 3.6)); axes = np.atleast_1d(axes)
for ax, d in zip(axes, present):
    a = read_csv_if(RES / "publishability_checks" / d / "ablation_study.csv")
    if a is None: ax.axis("off"); continue
    sc = ax.scatter(a["size_proxy_kb"], a["mcc"], s=50, color=DS_COLOR[d])
    for _, r in a.iterrows(): ax.annotate(r["model"].replace("final_params_", ""), (r["size_proxy_kb"], r["mcc"]), fontsize=6, xytext=(3, 3), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("model flash, exact export (KB, log)"); ax.set_ylabel("test MCC"); ax.set_title(DS_LABEL[d])
fig.suptitle("Ablation: feature count and pruning vs size"); save_fig(fig, FIG / "fig07_ablation"); plt.show()

# fig08 -- multi-seed
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
for ax, (k, lab) in zip(axes, [("mcc", "MCC"), ("macro_f1", "Macro-F1")]):
    for j, d in enumerate(present):
        ms = read_csv_if(RES / "article_results" / d / "multi_seed_repeats_summary.csv")
        if ms is None: continue
        r = ms[ms.iloc[:, 0] == k]
        ax.bar(j, float(r["mean"].iloc[0]), yerr=float(r["std"].iloc[0]), capsize=5, color=DS_COLOR[d]); ax.text(j, float(r["mean"].iloc[0]) + .03, f"{float(r['mean'].iloc[0]):.3f}", ha="center", fontsize=8)
    ax.set_xticks(range(len(present)), [DS_LABEL[d] for d in present], fontsize=7); ax.set_ylim(0, 1.05); ax.set_title(f"{lab} (mean ± std over seeds)")
save_fig(fig, FIG / "fig08_multi_seed"); plt.show()

In [ ]:
# fig09 / fig10 / fig11 -- optional figures (only if the corresponding notebooks were run)
import glob, shutil
ran = []
# fig09: convergence
hist = [(d, sorted(glob.glob(str(RES / "esp32_dt_final_results" / d / "search_history_*.csv")))) for d in present]
hist = [(d, f[-1]) for d, f in hist if f]
if hist:
    fig, axes = plt.subplots(1, len(hist), figsize=(5 * len(hist), 3.6)); axes = np.atleast_1d(axes)
    for ax, (d, f) in zip(axes, hist):
        h = pd.read_csv(f); bk = int(S[d]["validation_selected_k"]); h = h[h["k"] == bk].copy()
        h["o"] = h["stage"].map({"silverback": 0, "dolphin": 1, "prune": 2}); h = h.sort_values(["o", "eval_in_stage"]).reset_index(drop=True)
        h["eval"] = np.arange(1, len(h) + 1); h["best"] = h["fitness"].cummax()
        for st, c in [("silverback", "#4C72B0"), ("dolphin", "#55A868"), ("prune", "#C44E52")]:
            g = h[h.stage == st]; ax.scatter(g["eval"], g["fitness"], s=12, alpha=.5, color=c, label=st)
        ax.step(h["eval"], h["best"], where="post", color="black"); ax.set_title(f"{DS_LABEL[d]} (K={bk})"); ax.set_xlabel("evaluations"); ax.set_ylabel("CV fitness"); ax.legend(fontsize=7)
    save_fig(fig, FIG / "fig09_optimizer_convergence"); plt.show(); ran.append("fig09")
# fig10: tuning methods at equal budget (raw per-run results)
oc = [d for d in present if (RES / "optimizer_comparison" / d / "optimizer_comparison_raw.csv").exists()]
if oc:
    RAW = {d: pd.read_csv(RES / "optimizer_comparison" / d / "optimizer_comparison_raw.csv") for d in oc}
    has_runs = {d: ("deployable" in RAW[d].columns or "exported_kb" in RAW[d].columns) for d in oc}
    rows_n = 2 if any(has_runs.values()) else 1
    fig, axes = plt.subplots(rows_n, len(oc), figsize=(5.4 * len(oc), 3.9 * rows_n), squeeze=False)
    tabs, tabs_stats = [], []
    for j, d in enumerate(oc):
        r = RAW[d]; med = r.groupby("method")["val_mcc"].median().sort_values(); order = list(med.index)
        ax = axes[0, j]
        for i, m in enumerate(order):
            v = r[r.method == m]["val_mcc"].values
            ax.boxplot([v], positions=[i], vert=False, widths=.55, showfliers=False, patch_artist=True,
                       boxprops=dict(facecolor=HIGHLIGHT if m.startswith("Full") else "#cccccc", alpha=.5))
            ax.scatter(v, np.full(len(v), i) + np.random.default_rng(i).uniform(-.12, .12, len(v)), s=10, color=HIGHLIGHT if m.startswith("Full") else "#555", zorder=3)
        ax.set_yticks(range(len(order)), order, fontsize=7); ax.set_xlabel("validation MCC (one dot = one repeat)")
        ax.set_title(f"{DS_LABEL[d]}  (n = {r.repeat.nunique()} repeats)")
        if has_runs[d] and rows_n == 2:
            ax = axes[1, j]; summ = pd.read_csv(RES / "optimizer_comparison" / d / "optimizer_comparison_summary.csv", index_col=0)
            if "deployable_rate" in summ.columns:
                o2 = list(summ["deployable_rate"].sort_values().index); dr = summ["deployable_rate"].reindex(o2)
                lo, hi = summ["deployable_ci_low"].reindex(o2), summ["deployable_ci_high"].reindex(o2)
                ax.barh(o2, dr, xerr=[dr - lo, hi - dr], capsize=3, color=[HIGHLIGHT if m.startswith("Full") else "#bbbbbb" for m in o2])
                ax.set_xlim(0, 1.02); ax.set_yticks(range(len(o2)), o2, fontsize=7); ax.set_xlabel("deployable share of runs (95% CI)")
                tt = summ.reset_index()[["method", "n_runs", "val_mcc_median", "val_mcc_q25", "val_mcc_q75", "nodes_median", "exported_kb_median", "mean_rank", "deployable_rate"]]
                tt.insert(0, "Dataset", DS_LABEL[d]); tabs.append(tt)
            sf = RES / "optimizer_comparison" / d / "optimizer_comparison_stats.csv"
            if sf.exists():
                ss = pd.read_csv(sf); ss = ss[ss.metric == "val_mcc"][["other", "n_pairs", "mean_diff", "ci95_low", "ci95_high", "wilcoxon_p", "win_rate_proposed"]]
                ss.insert(0, "Dataset", DS_LABEL[d]); tabs_stats.append(ss)
        elif rows_n == 2:
            axes[1, j].axis("off")
    fig.suptitle("Optimizer comparison at equal evaluation budget (proposed = Full S→D→P)")
    fig.tight_layout(); save_fig(fig, FIG / "fig10_optimizer_comparison"); plt.show(); ran.append("fig10")
    if tabs:
        t7 = pd.concat(tabs); t7.columns = ["Dataset", "Method", "Runs", "Val MCC median", "Q25", "Q75", "Nodes (median)", "Exported KB (median)", "Mean rank", "Deployable rate"]
        save_table(t7, TAB / "tab07_optimizer_comparison", "Optimizer comparison at equal budget (median over repeats).", "tab:optcmp", "{:.3f}")
    if tabs_stats:
        t7b = pd.concat(tabs_stats); t7b.columns = ["Dataset", "Compared with", "Pairs", "Mean diff (proposed - other)", "CI95 low", "CI95 high", "Wilcoxon p", "Win rate"]
        save_table(t7b, TAB / "tab07b_optimizer_paired_stats", "Paired difference in validation MCC, proposed minus other.", "tab:optstats", "{:.4f}")
# fig11: class-weight ablation
cw = [d for d in present if (RES / "publishability_checks" / d / "class_weight_per_class.csv").exists()]
if cw:
    fig, axes = plt.subplots(1, len(cw), figsize=(5 * len(cw), 3.6)); axes = np.atleast_1d(axes)
    for ax, d in zip(axes, cw):
        c = read_csv_if(RES / "publishability_checks" / d / "class_weight_per_class.csv").pivot(index="class_name", columns="class_weight", values="recall").reindex(PC[d]["class_name"])
        c.plot.bar(ax=ax, width=.8); ax.set_ylim(0, 1.05); ax.set_title(DS_LABEL[d]); ax.set_xlabel(""); ax.set_ylabel("test recall"); plt.setp(ax.get_xticklabels(), rotation=60, ha="right")
    save_fig(fig, FIG / "fig11_class_weight_ablation"); plt.show(); ran.append("fig11")
    t8 = pd.concat([read_csv_if(RES / "publishability_checks" / d / "class_weight_ablation.csv").assign(Dataset=DS_LABEL[d]) for d in cw])
    save_table(t8[["Dataset", "class_weight", "val_mcc", "test_accuracy", "test_macro_f1", "test_mcc", "node_count"]], TAB / "tab08_class_weight", "Class-weight ablation.", "tab:cw", "{:.4f}")
# fig12: feature-policy ablation (notebook 07)
fpd = [d for d in present if (RES / "feature_policy_ablation" / d / "feature_policy_summary.csv").exists()]
if fpd:
    VORD = ["A_current_policy", "B_plus_ip_pattern_false_positives", "C1_plus_dst_port", "C2_plus_src_port", "C3_plus_both_ports", "D_old_policy_all", "E_ports_only"]
    fig, axes = plt.subplots(2, len(fpd), figsize=(5.4 * len(fpd), 6.6), squeeze=False); tabs9 = []
    for j, d in enumerate(fpd):
        sm = pd.read_csv(RES / "feature_policy_ablation" / d / "feature_policy_summary.csv"); dd = sm[sm.model == "default_DT"].set_index("variant")
        order = [v for v in VORD if v in dd.index]
        col = ["#1f77b4" if v.startswith("A_") else "#2ca02c" if v.startswith("B_") else "#d95f02" for v in order]
        for i, (m, sd, lab) in enumerate([("test_mcc", "test_mcc_std", "test MCC"), ("test_macro_f1", "test_macro_f1_std", "test macro-F1")]):
            ax = axes[i, j]; ax.bar(range(len(order)), dd.loc[order, m], yerr=dd.loc[order, sd].fillna(0), color=col, capsize=2)
            ax.set_xticks(range(len(order)), [o.split("_")[0] for o in order]); ax.set_ylabel(lab); ax.set_ylim(max(0, dd.loc[order, m].min() - .05), min(1.0, dd.loc[order, m].max() + .03))
            if i == 0: ax.set_title(DS_LABEL[d] + "  (default decision tree)")
        t = sm.copy(); t.insert(0, "Dataset", DS_LABEL[d]); tabs9.append(t[["Dataset", "model", "variant", "n_features", "test_mcc", "test_macro_f1", "minority_recall", "nodes", "port_importance_share"]])
    fig.suptitle("Feature-policy ablation: A leakage-safe features, B +'ip'-pattern features, C +ports, D A+B+ports, E ports only")
    fig.tight_layout(); save_fig(fig, FIG / "fig12_feature_policy"); plt.show(); ran.append("fig12")
    t9 = pd.concat(tabs9); t9.columns = ["Dataset", "Model", "Feature set", "Features", "Test MCC", "Test macro-F1", "Minority recall", "Nodes", "Port importance share"]
    save_table(t9, TAB / "tab09_feature_policy_ablation", "Feature-policy ablation (mean over repeats).", "tab:featpolicy", "{:.4f}")
print("optional figures produced:", ran or "none yet (run notebooks 01/02/04 first)")

# copy a few figures straight from the per-dataset folders (decision-tree drawings)
for d in present:
    src = RES / "figures" / d / "optimized_dt_top3_tree.png"
    if src.exists(): shutil.copy(src, FIG / f"tree_top3_{d}.png")